In [ ]:
pip install scikit-optimize

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from tensorflow.keras.regularizers import l2

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf

# scikit-optimize imports for Bayesian tuning
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt

In [2]:
# 2. NumPy
np.random.seed(42)

# 3. TensorFlow
tf.random.set_seed(42)

In [4]:
# Load data
df = pd.read_csv("/Users/Banjo/OneDrive/Desktop/MIDS/Summer 2025/DataSci 207/Final Project/w207FinalProject/data/processed/processed_plays.csv")

# Train/Validation/Test split (60/20/20)
train_val, test = train_test_split(df, test_size=0.20, random_state=42)
train, val      = train_test_split(train_val, test_size=0.20, random_state=42)

X_train, y_train = train.drop('yardsGained', axis=1), train['yardsGained']
X_val,   y_val   = val.drop('yardsGained', axis=1),   val['yardsGained']
X_test,  y_test  = test.drop('yardsGained', axis=1),  test['yardsGained']

In [5]:
df.head()

,down,yardsToGo,absoluteYardlineNumber,quarter,playAction,qbSneak,pff_runPassOption,yardsGained,secondsRemainingInQuarter,offFormation_EMPTY,...,passCoverage_Cover_3_Seam,passCoverage_Cover_6_Right,passCoverage_Goal_Line,passCoverage_Miscellaneous,passCoverage_Prevent,passCoverage_Quarters,passCoverage_Red_Zone,manZone_Man,manZone_Other,manZone_Zone
0,1,10,21,3,0,0,0,9,114,1,...,0,0,0,0,0,0,0,0,0,1
1,1,10,8,4,0,0,0,4,133,1,...,0,0,0,0,0,1,0,0,0,1
2,3,12,20,4,0,0,0,6,120,0,...,0,0,0,0,0,1,0,0,0,1
3,2,10,23,1,0,0,0,4,568,0,...,0,0,0,0,0,1,0,0,0,1
4,2,8,27,3,1,0,0,-1,136,0,...,0,0,0,0,0,0,0,1,0,0


In [41]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

input_dim = X_train_scaled.shape[1]
units = max(1, (500 - 1)//(input_dim + 2))
model = Sequential([
    Input(shape=(input_dim,)),
    Dense(units, activation='relu', kernel_regularizer=l2(1e-3)),
    Dense(1)
])


model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_27 (Dense)                │ (None, 7)              │           462 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 1)              │             8 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 470 (1.84 KB)

 Trainable params: 470 (1.84 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',  # Mean Squared Error
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'),
             tf.keras.metrics.MeanAbsoluteError(name='mae')]
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on train and validation sets
train_metrics = model.evaluate(X_train_scaled, y_train, verbose=0)
val_metrics   = model.evaluate(X_val_scaled, y_val, verbose=0)

print(f"\nNeural Net on train data → RMSE: {train_metrics[1]:.3f}  MAE: {train_metrics[2]:.3f}")
print(f"Neural Net on validation data → RMSE: {val_metrics[1]:.3f}  MAE: {val_metrics[2]:.3f}")

Epoch 1/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 74.4645 - mae: 5.7466 - rmse: 8.6224 - val_loss: 67.7160 - val_mae: 5.6118 - val_rmse: 8.2266
Epoch 2/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 866us/step - loss: 74.3547 - mae: 5.7404 - rmse: 8.6159 - val_loss: 67.7178 - val_mae: 5.6115 - val_rmse: 8.2267
Epoch 3/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 849us/step - loss: 74.3060 - mae: 5.7380 - rmse: 8.6131 - val_loss: 67.7427 - val_mae: 5.6127 - val_rmse: 8.2282
Epoch 4/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 848us/step - loss: 74.2453 - mae: 5.7356 - rmse: 8.6096 - val_loss: 67.7543 - val_mae: 5.6134 - val_rmse: 8.2289
Epoch 5/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 847us/step - loss: 74.1856 - mae: 5.7332 - rmse: 8.6061 - val_loss: 67.7741 - val_mae: 5.6152 - val_rmse: 8.2301
Epoch 6/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 863us/step - loss: 74.1332 - mae: 5.7311 - rmse: 8.6031 - val_loss: 67.8112 - val_mae: 5.6172 - val_rmse: 8.2324
Epoch 7/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 851us/step - los

In [49]:
def evaluate(name, actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    print(f"{name} → RMSE: {rmse:.3f}, MAE: {mae:.3f}, R²: {r2:.3f}")

evaluate('Train', y_train, model.predict(X_train_scaled).flatten())
evaluate('Validation', y_val,   model.predict(X_val_scaled).flatten())
evaluate('Test', y_test,  model.predict(X_test_scaled).flatten())


319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 404us/step
Train → RMSE: 8.712, MAE: 5.794, R²: 0.039
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 450us/step
Validation → RMSE: 8.227, MAE: 5.612, R²: 0.013
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 447us/step
Test → RMSE: 9.068, MAE: 5.898, R²: 0.026
